# 7000LBSBAI - Programming for Business Analytics
## Week 9 - Task: FitLife Over Time, by Segment, and a First Model

---

**Module:** Programming for Business Analytics  
**Week:** 9  
**Time:** About 75 minutes, plus optional extensions  
**Topics covered:** Creating an outcome from a date · Time series · Pivot tables of rates · Gemini as an assistant · Linear regression

---

### How to use this notebook

| Symbol | Meaning |
|--------|---------|
| ✍️ Worked example | Code is already written -> run it and study the output |
| ➡️ Your turn | Write or complete the code in this cell |
| 💬 What does it mean? | Write a short interpretation of your results |
| 🤖 AI activity | A guided activity using Gemini in Colab |
| 🔴 Extension | For students who finish early (optional) |
| 📝 Hint/Tip | Some friendly help and advice |

To run a cell: click on it and press **Shift + Enter**.

---

## ⚠️ BEFORE YOU CONTINUE

### Save a copy of this notebook

Make sure your notebook is saved to your Google Drive:
- **File → Save a copy in Drive**
- Name it: `Week9_Task_FitLife_Time_Segments_Regression_YourName.ipynb`

---

### ✨ Unhide generative AI features (keep inline completions turned off)

Go to **Tools** -> **Settings** -> **AI Assistance**

> Uncheck `☐ Show AI-powered inline completions`  
>
> Uncheck `☐ Hide generative AI features`

---
## The scenario

You are continuing your analysis of **FitLife Gyms** from Week 8. The membership manager now asks:

> ### *"How have joins and cancellations changed over time, which combinations of members are most at risk, and what drives how much members spend?"*

> 📝 Hints say which **code-along block** or **week** covers each skill. Some questions are open: decide how to investigate, then explain your choice.

---
## Set up: run this cell

This cell loads FitLife and cleans it with the Week 8 function. It also converts the dates and works out `tenure_months` (how long each member has belonged), as you did in Week 8.

In [ ]:
# ✍️ Set up: just run this cell
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf

sns.set_theme(style='whitegrid')

def clean_members(data):
    """Clean the FitLife member data (the Week 8 function)."""
    data = data.copy()
    data = data.drop_duplicates()
    data['membership_type'] = data['membership_type'].str.strip().str.title()
    data['monthly_fee'] = pd.to_numeric(data['monthly_fee'].str.replace('£', ''))
    data = data[(data['age'] >= 18) & (data['age'] <= 100)]
    data['visits_per_month'] = data['visits_per_month'].fillna(data['visits_per_month'].median())
    return data

df = clean_members(pd.read_csv("https://raw.githubusercontent.com/7000lbsbai/7000LBSBAI/main/datasets/fitlife_members.csv"))

df['join_date'] = pd.to_datetime(df['join_date'])
df['cancel_date'] = pd.to_datetime(df['cancel_date'])
end_date = df['cancel_date'].fillna(pd.Timestamp('2025-01-31'))
df['tenure_months'] = ((end_date - df['join_date']).dt.days / 30.44).round(1)

print(f"Data ready: {df.shape[0]} members")

---
# Section A: Create the outcome from a date
*about 10 minutes*

In real data, the outcome you want to study is not always a ready-made column. Here, a member has cancelled if they have a `cancel_date`. FitLife also has a `cancelled` column, so you can **check** your version against it.

> 📝 **Hint:** Week 8. `.isna()` is True where a value is missing.
>
> Use `df['cancelled_check'] == df['cancelled']).all()` to check each column against each other. You can also use `pd.crosstab(df['col1'],df['col2'])`

In [ ]:
# ➡️ A.1 Create a column 'cancelled_check': 'Yes' if the member has a cancel_date, 'No' if not. Then check it matches the cancelled column for every member


In [ ]:
# ➡️ A.2 Create a 1/0 column cancelled_flag from cancel_date, and print the cancellation rate (%)


### ✏️ A.3 Why check?

- Why is it important to **check** an outcome you have created yourself?
- What could go wrong if you **filled** the empty cancel dates before creating it?

**Your answer (double-click to edit):**

*Write your answer here.*

---
# Section B: Patterns over time
*about 20 minutes*

In [ ]:
# ➡️ B.1 Count the joins in each month and draw a line chart, with a title and axis labels


> 📝 **Hint:** code-along Block 1. `.size()` counts the rows in each month (there is no column to add up).

In [ ]:
# ➡️ B.2 Count the cancellations in each month (members who cancelled only), and draw a line chart with a 3-month rolling average


### ➡️ B.3 Is that spike real?

Something unusual happens in the **last month**. Before you report it, **investigate**: which **dates** have the most cancellations?

> 📝 **Hint:** `value_counts()` on `cancel_date`.

In [ ]:
# ➡️ B.3 Investigate the spike


### ✏️ B.4 What do you think happened?

- Is the January 2025 spike a **real** change in member behaviour? What is the evidence?
- What would you do about it before reporting monthly cancellations?

**Your answer (double-click to edit):**

*Write your answer here.*

In [ ]:
# ➡️ B.5 When do cancellations happen? For members who cancelled, count the cancellations in each month of membership (0, 1, 2…) and draw a bar chart


> 📝 **Hint:** `.astype(int)` turns 4.6 months into 4 (four complete months). **For Assessment 2:** NovaTel has `Tenure_Months`, so you can do the same kind of analysis there.

---
# Section C: Two factors at once
*about 15 minutes*

In Week 8, membership type and visits were both linked to cancellation. Do they **combine**?

1. Create `visits_band`: four equal-sized groups of `visits_per_month` (Week 8).
2. Create a pivot table of the **cancellation rate (%)** with `membership_type` as the rows and `visits_band` as the columns, and a second pivot table of the **counts**.
3. Draw a heatmap of the rates.

> 📝 **Hint:** code-along Block 2.

In [ ]:
# ➡️ C.1 A pivot table of cancellation rates by membership type and visits band, the counts, and a heatmap


### 💬 What does it mean?

- Which combination of members is most at risk, and which is least?
- Do membership type and visits **both** matter, even when you look at them together?
- Which group would you target first with a retention offer, and why?

**✏️ Your interpretation (double-click to edit):**

*Write your interpretation here.*

---
# Section D: 🤖 Gemini as an assistant
*about 15 minutes*

Last week Gemini was your **tutor**: it explained code. This week it is your **assistant**: it can **write** code for you, but **you** are responsible for checking it.

### This week's rules

| ✅ You may… | ⚠️ You must… |
|---|---|
| Ask Gemini to write code for a task you describe | Read and explain **every line** before you run it |
| Use the code if it is correct | **Check** the result against something you already know |
| | Never use code you cannot explain |
| | Record what happened in your AI log |

### The activity

1. Open the **Gemini** chat panel (✨). Make sure inline completions are still **off** (Week 8).

2. Paste this prompt:

> *"I have a pandas DataFrame called df. The column cancel_date has already been converted with pd.to_datetime, and it is empty for members who have not cancelled. Write Python code that counts the cancellations in each month and draws a line chart of the monthly counts with a 3-month rolling average, a title and axis labels. Use pandas and matplotlib only."*

3. Select ✔️ Accept. **Before running it**, check the `#` comment above each line saying what it does, and check you understand each section of code. If necessary, rewrite the `#` comment in your own words.

4. Run it, then **check** it in the next cell: does it give the same monthly counts as your answer to B.2?

In [ ]:
# 🤖 D.1 Gemini's code, with your own comment above each line


In [ ]:
# 🤖 D.2 Check: compare Gemini's monthly counts with yours from B.2


> 📝 **Checking ideas:** compare the totals (they should equal the number of members who cancelled), compare one or two months with B.2, and check that January 2025 shows the same spike.

### 📓 D.3 Add this to your AI log

Record what you asked, what Gemini produced, **how you checked it**, and anything it got wrong or did differently from you. If you started an AI log in Week 8, add it to this; if not, start an AI log now.

---
# Section E: Adding a regression model
*about 15 minutes*

What drives how much members **spend** each month? Start with one predictor: visits.

> 📝 **Hint:** code-along Block 4.

In [ ]:
# ➡️ E.1 Fit a model of monthly_spend explained by visits_per_month. Print the coefficient, its p-value and R², and predict the spend of members who visit 4 and 12 times a month


In [ ]:
# ➡️ E.2 Fit a second model with classes_per_month instead. Which single predictor explains more of the variation in spend?


### 💬 What does it mean?

- Explain the visits coefficient in plain English.
- What does an R² of about 0.24 tell you?
- Why might one predictor not be enough? What else could explain spend?

**✏️ Your interpretation (double-click to edit):**

*Write your interpretation here.*

---
# Section F: Summarise
*about 5 minutes*

### ✏️ F.1 Key findings

- Write **three** findings for the manager: one about time, one about combinations of members, and one about spend.

**Your answer (double-click to edit):**

*Write your answer here.*

---
## 🔴 Extension 1: which single predictor explains the most?

Use a **loop** to fit a one-predictor model of `monthly_spend` for each of these columns, and print each R²: `visits_per_month`, `classes_per_month`, `pt_sessions_per_month`, `distance_km`, `age`. Which explains the most? Which explains nothing?

In [ ]:
# 🔴 Extension 1: compare predictors with a loop


---
## 🔴 Extension 2: active members over time

How many members were **active** at the end of each month? A member is active at a date if they had joined by then and had **not yet cancelled**. Use a loop over the month-end dates.

In [ ]:
# 🔴 Extension 2: active members at the end of each month


---
## ✅ End of Task: Recap

You have taken your FitLife analysis further:

- **Created an outcome from a date**, and checked it: exactly the skill you need for NovaTel
- **Time series** of joins and cancellations, and **investigated a suspicious spike** before reporting it
- **Pivot tables** to see how two factors combine
- **Gemini as an assistant**: you generated code, explained it, and checked it
- Adding a **regression model**, and why one predictor is not enough

---

### 💾 Before you leave

Make sure your notebook is saved to your Google Drive, if you have not done so already:
- **File → Save a copy in Drive**
- Name it: `Week9_Task_FitLife_Time_Segments_Regression_YourName.ipynb`

---

### 🔮 Next week

Regression models with **several predictors**, and **logistic regression** for a yes/no outcome such as cancellation (or churn). Keep this notebook and your AI log.